In [0]:
#import required libraries
import datetime
import pyspark.sql.functions as f
import pyspark.sql.types
import pandas as pd

from pyspark.sql.functions import year, month, dayofmonth
from pyspark.sql.functions import unix_timestamp, from_unixtime
from pyspark.sql import Window
from pyspark.sql.functions import rank, min

import zipfile
import os

In [0]:
spark.conf.set(
    "fs.azure.sas.<Container-name>.<storage-account-name>.dfs.core.windows.net", 
    "<SAS-Token>"
)

In [0]:
#Access the files in the container
dbutils.fs.ls("abfss://<Container-name>@<storage-account-name>.dfs.core.windows.net/")

In [0]:
local_zip_path = "/tmp/ml-latest-small.zip"
extract_path = "/tmp/ml-latest-small"

In [0]:
# Copy the file from Azure Blob Storage to a local directory
dbutils.fs.cp("abfss://<Container-name>@<storage-account-name>.dfs.core.windows.net/ml-latest-small.zip", f"file:{local_zip_path}")

In [0]:
#Unzip the files
with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [0]:
# List extracted files
os.listdir(extract_path)

In [0]:
# Remove the ZIP file after extraction
os.remove(local_zip_path)

In [0]:
# Copy extracted CSV files back to Azure Blob Storage
dbutils.fs.cp(f"file:{extract_path}/movies.csv", "abfss://<Container-name>@<storage-account-name>.dfs.core.windows.net/movies.csv")
dbutils.fs.cp(f"file:{extract_path}/ratings.csv", "abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/ratings.csv")
dbutils.fs.cp(f"file:{extract_path}/tags.csv", "abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/tags.csv")
dbutils.fs.cp(f"file:{extract_path}/links.csv", "abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/links.csv")

In [0]:
# List files in Azure Blob Storage
dbutils.fs.ls("abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/")

In [0]:
# File locations for direct access
file_type = "csv"
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

In [0]:
%fs ls abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/

In [0]:
%fs ls abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/Source/ml-latest-small.zip/ml-latest-small/


In [0]:
%fs ls /FileStore/tables/

In [0]:
# Read Movies Data
movies_path = "abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/movies.csv"
df_movies = spark.read.format(file_type) \
      .option("inferSchema", infer_schema) \
      .option("header", first_row_is_header) \
      .option("sep", delimiter) \
      .load(movies_path)

display(df_movies)
df_movies.count()

In [0]:
# Read Links Data
links_path = "abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/links.csv"
df_links = spark.read.format(file_type) \
      .option("inferSchema", infer_schema) \
      .option("header", first_row_is_header) \
      .option("sep", delimiter) \
      .load(links_path)

display(df_links)


In [0]:
# Read Tags Data
tags_path = "abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/tags.csv"
df_tags = spark.read.format(file_type) \
      .option("inferSchema", infer_schema) \
      .option("header", first_row_is_header) \
      .option("sep", delimiter) \
      .load(tags_path)

display(df_tags)

In [0]:
# Read Ratings Data
ratings_path = "abfss://containerdatalake@samovielensanalysis.dfs.core.windows.net/ratings.csv"
df_ratings = spark.read.format(file_type) \
      .option("inferSchema", infer_schema) \
      .option("header", first_row_is_header) \
      .option("sep", delimiter) \
      .load(ratings_path)

display(df_ratings)

In [0]:
#first join movies and ratings tables
df_movies_with_ratings = df_movies.join(df_ratings,'movieId','left')

display(df_movies_with_ratings)

In [0]:
#Check if there are no duplicates
df_movies_no_dups = df_movies_with_ratings.groupby('movieId').count()
display(df_movies_no_dups)

In [0]:
df_movies.where(df_movies.movieId.isin([296,593, 858])).display()

In [0]:
#Join with users dataset
df_movies_with_ratings=df_movies_with_ratings.join(df_tags,['movieID','userId'],'inner')
display(df_movies_with_ratings)

In [0]:
#Join with ratings with tag
df_ratings_tags=df_ratings.join(df_tags,['movieID'],'inner')
display(df_movies_with_ratings)

In [0]:
df_ratings_add_tsdate=df_ratings.withColumn("tsdate",f.from_unixtime("timestamp"))

In [0]:
df_ratings_add_tsdate.display()

In [0]:
df_selected_rating_date=df_ratings_add_tsdate.select('userid','movieid','rating',f.to_date(unix_timestamp('tsdate','yyyy-MM-dd HH:mm:ss').cast('timestamp')).alias('rating_date'))
df_selected_rating_date.display()


In [0]:
df_rating_year=df_selected_rating_date.groupBy('rating_date').count()
df_rating_year.display()

In [0]:
df_avg_ratings=df_selected_rating_date.groupBy('movieid').mean('rating')
df_avg_ratings.display()

In [0]:
df=df_avg_ratings.join(df_movies,'movieid', 'inner')
df=df.withColumnRenamed('avg(rating)', 'avg_rating')
df.display()

In [0]:
df_total_rating=df_selected_rating_date.groupBy('movieid').count()
df_total_rating.display()

In [0]:
df_total_rating=df_total_rating.filter(df_total_rating['count']>50)
df_ratings_filtered=df_selected_rating_date.join(df_total_rating, 'movieid', 'inner')
df_ratings_filtered.display()

In [0]:
df_total_rating.count()

In [0]:
df_rating_per_user=df_ratings_filtered.select('userid','movieid','rating').groupBy('userid', 'movieid').max('rating')
df_rating_per_user_movie=df_rating_per_user.join(df_movies,'movieid','inner')
df_rating_per_user_movie=df_rating_per_user_movie.withColumnRenamed('max(rating)', 'max_rating')
df_rating_per_user_movie.display()

In [0]:
df_rating_max_rating=df_rating_per_user_movie.groupBy('userid','movieid','title','genres').max('max_rating')
df_rating_max_rating=df_rating_max_rating.withColumnRenamed('max(max_rating)', 'max_rating')
df_rating_max_rating.display()

In [0]:
df_rating_max_rating=df_rating_max_rating.filter(df_rating_max_rating['max_rating']>=4)
df_rating_max_rating.display()

In [0]:
#identify best movies per genre
df_movies_per_genere=df_rating_max_rating.groupBy('genres','title').count()
df_movies_per_genere.display()

In [0]:
#identify genres of user
df_ratings_genre=df_rating_max_rating.select('userid','title','genres').groupBy('userid','genres').count()
df_ratings_genre.display()

In [0]:
#latest Trending Movies Over all
df_recent_movie=df_ratings_filtered.groupBy('userid','movieid').agg(f.max(df_ratings_filtered['rating_date']))
df_recent_movie.display()

In [0]:
df_movies_per_genere=df.groupBy('genres').avg('avg_rating')
display(df_movies_per_genere)

In [0]:
# Create a view or table

temp_table_name = "movies_csv"

df_movies.createOrReplaceTempView(temp_table_name)

In [0]:
%sql

/* Query the created temp table in a SQL cell */

select * from `movies_csv`

In [0]:
%sql
DROP TABLE IF EXISTS default.permanent_table_name;

In [0]:
# With this registered as a temp view, it will only be available to this particular notebook. If you'd like other users to be able to query this table, you can also create a table from the DataFrame.
# Once saved, this table will persist across cluster restarts as well as allow various users across different notebooks to query this data.
# To do so, choose your table name and uncomment the bottom line.

permanent_table_name = "permanent_table_name"

df_movies.write.format("parquet").saveAsTable(permanent_table_name)

In [0]:
%sql

/* Query the created temp table in a SQL cell */

select * from `permanent_table_name`